In [1]:
import control
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.signal

sns.set_style("whitegrid")

Aircraft Pitch: PID Controller Design

Key MATLAB commands used in this tutorial are:
<http://www.mathworks.com/help/control/ref/controlsystemdesigner-app.html |controlSystemDesigner|>

From the main problem, the open-loop transfer function for the aircraft pitch dynamics is

$$ P(s) = \frac{\Theta(s)}{\Delta(s)} = \frac {1.151s+0.1774}{s^3+0.739s^2+0.921s} $$

where the input is elevator deflection angle $\delta$ and the output isthe aircraft pitch angle $\theta$.

For the original problem setup and the derivation of the above transfer function please refer to the
< ?example=AircraftPitch&section=SystemModeling Aircraft Pitch: System Modeling> page. 

For a step reference of 0.2 radians, the design criteria are thefollowing.

* Overshoot less than 10%
* Rise time  less than 2 seconds
* Settling time less than 10 seconds
* Steady-state error less than 2% 

Recall from the 
< ?example=Introduction&section=ControlPID Introduction: PID Controller Design> page 
that the transfer function for a PID controller is the following. 

$$ C(s) = K_{p} + \frac {K_{i}}{s} + K_{d}s = \frac{K_{d}s^2 + K_{p}s +
K_{i}}{s} $$

We will implement combinations of proportional ($K_p$), integral ($K_i$), and derivative ($K_d$)
control in the unity-feedback architecture shown below in order to achieve the desired system behavior.

![feedback_pitch2.png](figures/feedback_pitch2.png)

In particular, we will take advantage of the automated tuningcapabilities of the *Control System Designer* within MATLAB to design our PIDcontroller. First, enter the following code at the command line todefine the model of our plant $P(s)$. Refer to the
< ?example=AircraftPitch&section=SystemModeling Aircraft Pitch: System Modeling> page 
for the details of getting these commands.

In [2]:

s = control.tf('s')
P_pitch = (1.151*s+0.1774)/(s**3+0.739*s**2+0.921*s)


Proportional control
Let's begin by designing a proportional controller of the form $C(s) = K_p$.
The *Control System Designer* we will use for design can be opened by typing
|controlSystemDesigner(P_pitch)| at the command line. The *Control System
Designer* window will initially open with the root locus plot, open-loop Bodeplot, and closed-loop step response plot displayed for the provided plant  
transfer function with controller $C(s) = 1$, by default.

The *Edit Architecture* button in *Control System Designer* window displays thearchitecture of the control system being designed as shown below. Thisdefault agrees with the architecture we are employing.

![Picture1.png](figures/Picture1.png)

Since our reference is a step function of 0.2 radians, we can set theprecompensator block $F(s)$ equal to 0.2 to scale a unit step input to oursystem. This can be accomplished from the *Compensator Editor* window,
which can be opened by right-clicking on the plot and thenselecting *Edit Compensator*.
Specifically, choose |F| from the drop-down menu in the
*Compensator* portion of the window and set the compensator equal to 0.2
as shown in the figure below.

![Picture2.png](figures/Picture2.png)

To begin with, let's see how the system performs with a proportional controller $K_p$ set equal to 2. The compensator $C(s)$ can be defined in the same manner as the precompensator, just choose |C| from the drop-down menu in the *Compensator* portion of the window instead of |F|. Then set the compensator equal to 2. To see the performance of our system with this controller, move to the *IOTransfer_r2y:step* tab. If you have accidentally closed this tab, you can re-open it from the *Control System Designer* window by clicking on the *New Plot* menu and selecting *New Step*. In response, a new window titled *New Step to plot* will appear. From the *Select Responses to  Plot* menu, then choose *IOTransfer_r2y* and click the button *Plot* as shown below.


![Picture3.png](figures/Picture3.png)


A window will then open with the following step response displayed.


![Picture4.png](figures/Picture4.png)


Examination of the above shows that aside from steady-state error, the given design requirements have not been met. The gain chosen for $K_p$ can be adjusted in an attempt to modify the resulting performance through the *Compensator Editor* window. Instead, we will use the *Control System Designer* to automatically tune our proportional compensator. In order to use this feature, go to the *Tuning Methods* menu of the MATLAB toolstrip and choose *PID Tuning* under the *Automated Tuning* menu. Then select a *Controller type* of |P| and *Select  Loop to Tune* as |LoopTransfer_C| as shown in the figure below (our architecture has only one loop).


![Picture5.png](figures/Picture5.png)


There are two options that can be chosen from the *Tuning method* drop-down menu, |Robust response time| or |Classical design formulas|. The |Robust response time| algorithm automatically tunes  the PID parameters to balance speed of response and robustness. It can tune all parameters for any type of PID controller. It can be used for design of plants that are stable, unstable, or integrating. However, the |Classical design formulas| algorithm requires a stable or integrating plant and cannot tune the derivative filter. If you select |Classical design formulas| algorithm,  then in the *Formula* drop-down menu a range of options can be seen. These options range from heuristic techniques, like Ziegler-Nichols, to  numerical approaches that search over all possible control gains to minimize  some identified performance index. For our example, choose the |Robust response time| algorithm. Then in the *Design mode* drop-down menu, you can choose |Time| or |Frequency|. Since our design requirements are expressed in the         time-domain, we select the *Design mode* as |Time|. Since, our rise time is expected to be less than 2 seconds, try specifying a *Response Time* of 1.5 seconds.


Once all of the tuning settings have been chosen, then click the *Update Compensator* button. The algorithm then chooses a proportional gain of $K_p$ = 1.1269. This controller meets the rise time requirement, but the settle time is much too large. You can attempt requiring a faster response time (move the slider to the right), however, this will result in an increase in overshoot and oscillation. The proportional controller does not provide us a sufficient degree of freedom in our tuning, we need to add integral and/or derivative terms to our controller in order to meet the given requirements.


![Picture6.png](figures/Picture6.png)


PI control Recalling the information provided in the  < ?example=Introduction&section=ControlPID Introduction: PID Controller Design> tutorial,  integral control is often helpful in reducing steady-state error. In our case, the steady-state error requirement is already met. For purposes of illustration, let's design a PI controller anyway. We will again use automated tuning to choose our controller gains as we did above, only now we will select a *Controller type* of |PI|. Everything else will be left unchanged. Clicking on the *Compensator Update* button then produces the following controller.


$$ C(s) = 0.026294\frac{1 + 43s}{s} \approx \frac{0.0263}{s} + 1.13 $$


This transfer function is a PI compensator with $K_i$ = 0.0263 and $K_p$ = 1.13. The resulting closed-loop step response is shown below.


![Picture8.png](figures/Picture8.png)


From inspection of the above, notice that the addition of integral control helped reduce the average error in the signal more quickly (the error changes sign around 20 seconds), but it didn't help reduce the oscillation. Let's try also adding a derivative term to our controller. P

ID Control  Again recalling the lessons we have learned in the  < ?example=Introduction&section=ControlPID Introduction: PID Controller Design> tutorial,  increasing the derivative gain $K_d$ in a PID controller can often help reduce overshoot. Therefore, by adding derivative control we may be able to reduce the oscillation in the response a sufficient amount that we can then increase the other gains to reduce the settling time. Let's test our hypothesis by changing the *Controller type* to |PID| and again click the *Update Compensator* button. The generated controller is shown below.


$$ C(s) = 0.5241 \frac{(1 + 1s)( 1 + 1s)}{s} \approx \frac{0.5241}{s} + 1.0482 + 0.5241s $$


This transfer function is a PID compensator with $K_i$ = 0.5241, $K_p$ = 1.0482, and $K_d$ = 0.5241. The resulting closed-loop step response is shown below.


![Picture9.png](figures/Picture9.png)


This response meets all of the requirements except for the settle time which at 19.7 seconds is  larger than the given requirement of 10 seconds. Reducing the *Response Time* requirement (moving the slider to the right) will make the response faster, while moving the *Transient Behavior* slider towards *Robust* will help reduce the oscillation. The resulting PID controller for the shown settings is given below.


$$ C(s) = 1.2882 \frac{1 + 3.24s +0.2016s^2}{s} \approx \frac{1.2882}{s} + 4.17 + 0.26s $$


![Picture10.png](figures/Picture10.png)


Here we can see that moving both sliders to the right made the response faster and reduced the oscillation. However, the settling time is still greater than the required 10 seconds. We again try increasing the required speed of response; we have some room to spare on overshoot. The resulting PID controller for the settings shown below is the following.


$$ C(s) = 1.74 \frac{1 + 2.98s +1.716s^2}{s} \approx \frac{1.74}{s} + 5.1852 + 2.98s $$


![Picture11.png](figures/Picture11.png)


This response meets all of the given requirements as summarized below.


* Overshoot = 7.5% < 10% * Rise time = 0.413 seconds < 2 seconds * Settling time = 9.25 seconds < 10 seconds * Steady-state error = 0% < 2%


Therefore, this PID controller will provide the desired performance of the aircraft's pitch.
